In [6]:
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("OPENAI_KEY")

In [10]:
import os
import streamlit as st
import langchain_core
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


# Load OpenAI API key — prefer environment variable or .env file over hardcoding
os.environ['OPENAI_API_KEY'] = 'your-openai-api-key-here'

# (1) Initialise LLM with current class (ChatOpenAI replaces the legacy OpenAI completer)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9, max_tokens=500, api_key=api_key)

# (2) Load data from URLs
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load()
print(f"Loaded {len(data)} documents")

# (3) Split data into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
docs = text_splitter.split_documents(data)
print(f"Split into {len(docs)} chunks")

# (4) Create embeddings and build FAISS index
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key)
vectorstore = FAISS.from_documents(docs, embeddings)

# Save/load using FAISS native persistence (replaces pickle)
index_path = "vector_index"
vectorstore.save_local(index_path)

# Load back from disk
vectorstore = FAISS.load_local(
    index_path,
    embeddings,
    allow_dangerous_deserialization=True  # required flag for local FAISS loads
)

# (5) Build a RAG chain using modern LCEL (replaces RetrievalQAWithSourcesChain)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context.
If the answer is not in the context, say "I don't have enough information to answer this."

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    )

chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# (6) Enable debug logging (replaces langchain.debug = True)
langchain_core.globals.set_debug(True)

# (7) Run a query
query = "what is the price of Tiago iCNG?"
result = chain.invoke(query)
print(result)

Loaded 2 documents
Split into 11 chunks
[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question>] Entering Chain run with input:
{
  "input": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence] Entering Chain run with input:
{
  "input": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "what is the price of Tiago iCNG?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<context,question> > chain:RunnableSequence

Using Mapping Retrival Approach

In [14]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.globals import set_debug

os.environ['OPENAI_API_KEY'] = 'your-openai-api-key-here'

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.9, max_tokens=500, api_key=api_key)

# (1) Load data
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load()

# (2) Split into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(data)

# (3) Build and persist FAISS index
embeddings = OpenAIEmbeddings(model="text-embedding-3-small", api_key=api_key)
vectorstore = FAISS.from_documents(docs, embeddings)

index_path = "vector_index"
vectorstore.save_local(index_path)
vectorstore = FAISS.load_local(index_path, embeddings, allow_dangerous_deserialization=True)

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# (4) MAP prompt — called once per retrieved chunk
map_prompt = ChatPromptTemplate.from_template("""
You are answering a question using a single excerpt from an article.
Extract any relevant information from the excerpt to help answer the question.
If the excerpt contains no relevant information, respond with "No relevant info."

Article excerpt (source: {source}):
{context}

Question: {question}

Relevant information:
""")

# (5) REDUCE prompt — combines all mapped answers into a final response
reduce_prompt = ChatPromptTemplate.from_template("""
You are synthesizing partial answers from multiple article excerpts into one final answer.
Use only the information provided. If the answer isn't present, say so clearly.

Partial answers from each excerpt:
{mapped_answers}

Question: {question}

Final answer:
""")

# (6) Map step — run LLM individually on each retrieved chunk
map_chain = map_prompt | llm | StrOutputParser()

def map_docs(inputs: dict) -> dict:
    question = inputs["question"]
    retrieved_docs = retriever.invoke(question)
    
    mapped = []
    for doc in retrieved_docs:
        result = map_chain.invoke({
            "context": doc.page_content,
            "source": doc.metadata.get("source", "unknown"),
            "question": question
        })
        mapped.append(f"[Source: {doc.metadata.get('source', 'unknown')}]\n{result}")
    
    return {
        "mapped_answers": "\n\n---\n\n".join(mapped),
        "question": question
    }

# (7) Reduce step — synthesize mapped answers into final response
reduce_chain = reduce_prompt | llm | StrOutputParser()

# (8) Full map-reduce chain
map_reduce_chain = RunnableLambda(map_docs) | RunnableLambda(
    lambda x: reduce_chain.invoke(x)
)

# (9) Run a query
set_debug(True)

query = "what is the price of Tiago iCNG?"
result = map_reduce_chain.invoke({"question": query})
print(result)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "question": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:map_docs] Entering Chain run with input:
{
  "question": "what is the price of Tiago iCNG?"
}
[chain/start] [chain:RunnableSequence > chain:map_docs > chain:RunnableSequence] Entering Chain run with input:
{
  "context": "Eco Pulse\n\nMC Learn\n\nTrending Topics\n\nSensex Live\n\nLenskart Block Deal\n\nGift Nifty\n\nPetrol and Diesel Prices Today\n\nOnEMI Technology IPO\n\nTata Motors launches Punch iCNG, price starts at Rs 7.1 lakh\n\nThe Punch iCNG is equipped with the company's proprietary twin-cylinder technology with enhanced safety features like a micro-switch to keep the car switched off at the time of refuelling and thermal incident protection that cuts off CNG supply to the engine and releases gas into the atmosphere, Tata Motors said in a statement.\n\nPTI\n\nAugust 04, 2023 / 14:17 IST\n\njoin Us On WhatsApp\